# Convex Hull & Ordering — FP Job Generator

Generates HPC job scripts, submission scripts, and merge scripts for running
one or more foundation potentials (FPs) against the standardized DFT
reference, producing output that feeds directly into the existing analysis
notebook with no conversion step:

```text
standardized DFT reference
        |
generate jobs for one FP or all FPs
        |
submit one FP or all FPs
        |
merge each FP's completed chunks
        |
merge all FP fragments
        |
phase_stability_ordering_results_standardized.json.gz
        |
existing analysis notebook (convexhull_ordering_analysis_all_models.ipynb)
```

This notebook only **generates and validates scripts** -- it never submits
SLURM jobs, imports any FP package, loads any model, or runs any calculator.
All of that happens later, on Zaratan, by running the generated scripts.

**Sections**
| # | Title |
|---|-------|
| 0 | Imports & setup |
| 1 | Load and validate the standardized DFT reference |
| 2 | Potential registry (Zaratan paths) |
| 3 | Configuration |
| 4 | Script building blocks (SLURM template, energy function, script header) |
| 5 | Generate hull jobs (interior + endpoint candidates, combined) |
| 6 | Generate ordering jobs |
| 7 | Per-FP and all-FP submission scripts |
| 8 | Merge completed chunks and FP fragments |
| 9 | Finalize and merge completed results |


---
## Section 0 — Imports & setup

In [ ]:
import os
import sys
import json
import gzip
import stat
import math
import glob
import random
import hashlib
import textwrap
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np

REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))

from convexhull_analysis_utils import (
    load_standardized_reference,
    load_standardized_results,
    sha256_of_file,
    write_standardized_json_gz,
    merge_phase_stability_ordering_fp_results,
    build_phase_stability_ordering_results,
    validate_phase_stability_ordering_results,
    PHASE_STABILITY_ORDERING_SCHEMA_VERSION,
    PHASE_STABILITY_ORDERING_DATASET_NAME,
)

print("Imports OK.")
print(f"schema_version = {PHASE_STABILITY_ORDERING_SCHEMA_VERSION!r}")
print(f"dataset_name   = {PHASE_STABILITY_ORDERING_DATASET_NAME!r}")

---
## Section 1 — Load and validate the standardized DFT reference

Loads only `data/phase_stability_ordering_reference.json.gz` -- the single
canonical DFT source for job generation. Does **not** read
`ALL_TIELINES_DATA.json`, `binary_entries/`, or use a `TOP_K` cutoff: the
reference file already carries the exact, final tie-line-system / interior /
endpoint / ordering-group population (candidate selection is not redone
here), each candidate's natural identifiers, its raw uncorrected DFT-PBE
`energy_total`, its DFT-PBE `relaxed_structure`, and its `initial_structure`
(the un-relaxed starting geometry for full FP relaxation -- for binary
endpoints this is the same structure as `relaxed_structure`, since only one
DFT reference structure exists for them; see
`reference_metadata["endpoint_initial_structure_convention"]`).

Before generating any job, every required identifier, energy, and structure
is confirmed present. If anything required is missing, this stops with a
clear list of the exact affected candidates rather than generating a job
with invented or substituted data.

In [ ]:
REFERENCE_PATH = Path("data/phase_stability_ordering_reference.json.gz")

reference_payload = load_standardized_reference(REFERENCE_PATH)
REFERENCE_SHA256 = sha256_of_file(REFERENCE_PATH)
reference_data = reference_payload["reference_data"]

print(f"Loaded {REFERENCE_PATH} (sha256={REFERENCE_SHA256[:12]}...)")
print(f"schema_version: {reference_payload['schema_version']}  dataset_name: {reference_payload['dataset_name']}")
print(f"reference_population: {reference_payload['reference_metadata']['reference_population']}")

In [ ]:
def _validate_reference(reference_data):
    """Confirm every required identifier, energy, and structure is present
    for every hull and ordering candidate. Raises ValueError listing the
    exact affected candidates on any gap -- never substitutes or invents a
    fallback. Also confirms the locked-in reference population counts."""
    problems = []

    hull = reference_data.get("hull", {})
    n_interior = n_endpoints_appearances = 0
    unique_endpoints = set()
    for system, candidates in hull.items():
        for cid, rec in candidates.items():
            for field in ("role", "phase_id", "composition", "energy_total", "relaxed_structure", "initial_structure"):
                if rec.get(field) is None:
                    problems.append(f"hull[{system!r}][{cid!r}] missing required field {field!r}")
            if rec.get("role") == "interior":
                n_interior += 1
            elif rec.get("role") == "endpoint":
                n_endpoints_appearances += 1
                unique_endpoints.add(cid)
                if rec.get("endpoint_side") not in ("left", "right"):
                    problems.append(f"hull[{system!r}][{cid!r}] endpoint missing endpoint_side")

    ordering = reference_data.get("ordering", {})
    n_ordering_records = 0
    for group_key, group in ordering.items():
        orderings = group.get("orderings", {})
        if len(orderings) != 20:
            problems.append(f"ordering[{group_key!r}] has {len(orderings)} orderings, expected 20")
        for name, rec in orderings.items():
            n_ordering_records += 1
            for field in ("energy_total", "relaxed_structure", "initial_structure"):
                if rec.get(field) is None:
                    problems.append(f"ordering[{group_key!r}]['orderings'][{name!r}] missing required field {field!r}")

    if problems:
        raise ValueError(
            f"Reference validation failed with {len(problems)} problem(s); "
            "refusing to generate jobs from an incomplete reference. First 20:\n  "
            + "\n  ".join(problems[:20])
        )

    return {
        "n_systems": len(hull),
        "n_interior_candidates": n_interior,
        "n_unique_endpoints": len(unique_endpoints),
        "n_total_hull_candidates": n_interior + len(unique_endpoints),
        "n_ordering_groups": len(ordering),
        "n_ordering_records": n_ordering_records,
    }


population = _validate_reference(reference_data)
print("Validation OK:", population)

EXPECTED_POPULATION = {
    "n_systems": 22,
    "n_interior_candidates": 561,
    "n_unique_endpoints": 36,
    "n_total_hull_candidates": 597,
    "n_ordering_groups": 305,
    "n_ordering_records": 6100,
}
mismatched = {k: (population[k], v) for k, v in EXPECTED_POPULATION.items() if population[k] != v}
if mismatched:
    raise ValueError(f"Reference population does not match the locked-in manuscript counts: {mismatched}")
print("Population matches the locked-in manuscript counts exactly.")

---
## Section 2 — Potential registry (Zaratan paths)

Every entry's `site_pkgs`, `venv_activate`, `python`, `model_path`, and
`calc_setup` is the exact value already established for this project's real
Zaratan environments -- none of these are corrected, normalized, or guessed
here, including `"tensornet"`'s r2SCAN model path (kept exactly as-is; it is
a different, still-valid registry entry from the newly added
`"tensornet_pbe"`).

`m3gnet_mp` and `tensornet_pbe` were not previously registered under those
keys. Their `site_pkgs`/`venv_activate`/`python` are identical to the
already-registered `m3gnet_matpes_pbe`/`tensornet` entries (same Zaratan
environment), and their `model_path`/`calc_setup` are taken verbatim from
this project's own real production job scripts that generated the
manuscript's existing M3GNet and TensorNet-MatPES-PBE results
(`Felix_convex_hull/running_codes_convex_hull/run_0_m3gnet_pes.py`,
`run_0_tensornet_matpes_pbe.py`, and their ordering-side counterparts) --
not invented.

In [ ]:
def _require_configured(value, name):
    s = str(value)
    if value is None or s.startswith("/path/to/") or s.startswith("YOUR_"):
        raise ValueError(
            f"{name} has not been configured -- edit this cell and set {name} to "
            "the real value for your system before running this notebook."
        )
    return value


POTENTIAL_REGISTRY = {

    # ── CHGNet ────────────────────────────────────────────────────────────────
    "chgnet": {
        "mlip_name":     "chgnet",
        "site_pkgs":     "/path/to/chgnet_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/chgnet_env/bin/activate",
        "python":        "/path/to/chgnet_env/bin/python",
        "model_path":    None,   # CHGNetCalculator() loads its default checkpoint from the package itself
        "calc_setup":    textwrap.dedent("""
            from chgnet.model.dynamics import CHGNetCalculator
            calc = CHGNetCalculator()
        """).strip(),
    },

    # ── MACE (MPA-0 medium) ───────────────────────────────────────────────────
    "mace": {
        "mlip_name":     "mace",
        "site_pkgs":     "/path/to/mace_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/mace_env/bin/activate",
        "python":        "/path/to/mace_env/bin/python",
        "model_path":    "/path/to/checkpoints/mace-mpa-0-medium.model",
        "calc_setup":    textwrap.dedent("""
            from mace.calculators import MACECalculator
            calc = MACECalculator(
                model_paths=[MODEL_PATH],
                device="cpu",
                default_dtype="float64",
            )
        """).strip(),
    },

    # ── M3GNet (MatPES-PBE v2025.1) ───────────────────────────────────────────
    "m3gnet_matpes_pbe": {
        "mlip_name":     "m3gnet_matpes_pbe",
        "site_pkgs":     "/path/to/m3gnet_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/m3gnet_env/bin/activate",
        "python":        "/path/to/m3gnet_env/bin/python",
        "model_path":    "/path/to/checkpoints/M3GNet-MatPES-PBE-v2025.1-PES",
        "calc_setup":    textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── M3GNet (MP, 2021.2.8 -- the plain, MatPES-independent M3GNet baseline) ─
    # Uses the same environment as "m3gnet_matpes_pbe" (MatGL); only the
    # checkpoint differs.
    "m3gnet_mp": {
        "mlip_name":     "m3gnet_mp",
        "site_pkgs":     "/path/to/m3gnet_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/m3gnet_env/bin/activate",
        "python":        "/path/to/m3gnet_env/bin/python",
        "model_path":    "/path/to/checkpoints/M3GNet-MP-2021.2.8-PES",
        "calc_setup":    textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── TensorNet (MatPES r2SCAN v2025.1) ─────────────────────────────────────
    "tensornet": {
        "mlip_name":     "tensornet",
        "site_pkgs":     "/path/to/m3gnet_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/m3gnet_env/bin/activate",
        "python":        "/path/to/m3gnet_env/bin/python",
        "model_path":    "/path/to/checkpoints/TensorNet-MatPES-r2SCAN-v2025.1-PES",
        "calc_setup":    textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── TensorNet (MatPES-PBE v2025.1) ────────────────────────────────────────
    # Uses the same environment as "tensornet" (MatGL); only the checkpoint differs.
    "tensornet_pbe": {
        "mlip_name":     "tensornet_pbe",
        "site_pkgs":     "/path/to/m3gnet_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/m3gnet_env/bin/activate",
        "python":        "/path/to/m3gnet_env/bin/python",
        "model_path":    "/path/to/checkpoints/TensorNet-MatPES-PBE-v2025.1-PES",
        "calc_setup":    textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── UMA (FAIRChem / Meta) ─────────────────────────────────────────────────
    "uma": {
        "mlip_name":     "uma",
        "site_pkgs":     "/path/to/uma_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/uma_env/bin/activate",
        "python":        "/path/to/uma_env/bin/python",
        "model_path":    "/path/to/checkpoints/uma-s-1p1.pt",
        "calc_setup":    textwrap.dedent("""
            from fairchem.core import FAIRChemCalculator
            from fairchem.core.units.mlip_unit import load_predict_unit
            calc = FAIRChemCalculator(
                load_predict_unit(MODEL_PATH, device="cpu"),
                task_name="omat",
            )
        """).strip(),
    },

    # ── MACE fine-tuned (Tong, C128, N=1200) ──────────────────────────────────
    "mace_ft_c128_n1200": {
        "mlip_name":     "mace_ft_c128_n1200",
        "site_pkgs":     "/path/to/mace_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/mace_env/bin/activate",
        "python":        "/path/to/mace_env/bin/python",
        "model_path":    "/path/to/checkpoints/energy+forces_model_C128_L3_R60_N1200_compiled.model",
        "calc_setup":    textwrap.dedent("""
            from mace.calculators import MACECalculator
            calc = MACECalculator(
                model_paths=[MODEL_PATH],
                device="cpu",
                default_dtype="float64",
            )
        """).strip(),
    },

    # ── MACE-MatPES-PBE (omat fine-tuned) ─────────────────────────────────────
    "mace_matpes_pbe": {
        "mlip_name":     "mace_matpes_pbe",
        "site_pkgs":     "/path/to/mace_env/lib/python3.10/site-packages",
        "venv_activate": "/path/to/mace_env/bin/activate",
        "python":        "/path/to/mace_env/bin/python",
        "model_path":    "/path/to/checkpoints/MACE-matpes-pbe-omat-ft.model",
        "calc_setup":    textwrap.dedent("""
            from mace.calculators import MACECalculator
            calc = MACECalculator(
                model_paths=[MODEL_PATH],
                device="cpu",
                default_dtype="float64",
            )
        """).strip(),
    },
}

# The seven FPs evaluated on MatPES-PBE in the manuscript. Registry keys here
# are already the canonical model keys build_phase_stability_ordering_results
# / merge_phase_stability_ordering_fp_results expect (see
# convexhull_analysis_utils.PHASE_STABILITY_ORDERING_MODEL_ORDER) -- no
# separate legacy-to-canonical remapping is needed.
MATPES_PBE_POTENTIALS = [
    "mace",
    "chgnet",
    "m3gnet_mp",
    "uma",
    "m3gnet_matpes_pbe",
    "tensornet_pbe",
    "mace_matpes_pbe",
]

for key in MATPES_PBE_POTENTIALS:
    assert key in POTENTIAL_REGISTRY, f"{key!r} missing from POTENTIAL_REGISTRY"

print("Registered potentials:", list(POTENTIAL_REGISTRY.keys()))
print("MatPES-PBE potentials (the seven evaluated in the manuscript):", MATPES_PBE_POTENTIALS)
print("Every path above is a placeholder -- edit POTENTIAL_REGISTRY for the FP(s) you "
      "intend to actually run before generating jobs (Section 5 validates this for you).")

---
## Section 3 — Configuration

`SELECTED_POTENTIALS` controls whether this run generates jobs for one FP or
all seven:

```python
SELECTED_POTENTIALS = MATPES_PBE_POTENTIALS   # all seven
# or
SELECTED_POTENTIALS = ["mace"]                # a single FP
```

Full FP relaxation always uses fixed-cell `FIRE` (atomic positions only,
cell shape/volume held fixed) with `fmax = 0.01 eV/A` and up to `10,000`
steps -- the same optimizer, convergence criterion, and step limit already
used for every FP's existing manuscript results (see
`Felix_convex_hull/running_codes_convex_hull/run_0_*.py` and
`running_codes_ordering/job_000_*.py`: every one of them uses plain
`FIRE(atoms, logfile=None)`, no variable-cell filter). A relaxation that
reaches the step limit without reaching `fmax` is recorded as
`non_converged`, not `success`.

In [ ]:
# ── Which FPs to generate jobs for ──────────────────────────────────────────
SELECTED_POTENTIALS = MATPES_PBE_POTENTIALS   # all seven; or e.g. ["mace"]

for key in SELECTED_POTENTIALS:
    assert key in POTENTIAL_REGISTRY, f"{key!r} not in POTENTIAL_REGISTRY"

# ── Protocol settings (fixed-cell full relaxation + static evaluation) ─────
FMAX      = 0.01     # eV/Angstrom
MAX_STEPS = 10000
FIXED_CELL = True    # atomic positions only; cell shape/volume held fixed --
                      # matches every existing production job script exactly

# ── Chunking ─────────────────────────────────────────────────────────────────
CHUNK_SIZE = 50       # candidates per chunk JSON / SLURM job

# ── Output root (local, relative -- created here; no Zaratan access needed) ──
OUTPUT_ROOT = Path("generated_jobs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUBMITTED_JOBS_TSV = OUTPUT_ROOT / "submitted_jobs.tsv"

# ── SLURM settings -- EDIT for your cluster ─────────────────────────────────
SLURM = {
    "account":       "YOUR_SLURM_ACCOUNT",   # EDIT: your cluster allocation/account
    "partition":     "YOUR_SLURM_PARTITION", # EDIT: your cluster's partition/queue name
    "ntasks":        1,
    "cpus_per_task": 2,
    "mem_per_cpu":   "16G",
    "time":          "48:00:00",
}
_require_configured(SLURM["account"], "SLURM['account']")
_require_configured(SLURM["partition"], "SLURM['partition']")

# Confirm every selected FP's registry entry has actually been edited before
# generating any job for it -- catches a forgotten placeholder early rather
# than generating scripts that will fail on Zaratan.
for _fp_key in SELECTED_POTENTIALS:
    _cfg = POTENTIAL_REGISTRY[_fp_key]
    _require_configured(_cfg["site_pkgs"], f"POTENTIAL_REGISTRY[{_fp_key!r}]['site_pkgs']")
    _require_configured(_cfg["venv_activate"], f"POTENTIAL_REGISTRY[{_fp_key!r}]['venv_activate']")
    _require_configured(_cfg["python"], f"POTENTIAL_REGISTRY[{_fp_key!r}]['python']")
    if _cfg["model_path"] is not None:
        _require_configured(_cfg["model_path"], f"POTENTIAL_REGISTRY[{_fp_key!r}]['model_path']")

print(f"Selected potentials : {SELECTED_POTENTIALS}")
print(f"fmax / max_steps    : {FMAX} / {MAX_STEPS}  (fixed_cell={FIXED_CELL})")
print(f"Chunk size          : {CHUNK_SIZE}")
print(f"Output root         : {OUTPUT_ROOT.resolve()}")

---
## Section 4 — Script building blocks

`ENERGY_FUNC` is embedded verbatim into every generated per-chunk job
script. Full FP relaxation and static FP evaluation each get their own
`try`/`except` in the per-candidate loop (Sections 5-6) so a failure in one
protocol never prevents or erases the other. Recorded per candidate/protocol:
`status` (`success`/`failed`/`non_converged`), `energy_total` (eV),
`forces`, the FP-relaxed structure (relax only), `converged`, `n_steps`,
`fmax_requested`, `max_steps`, and `fixed_cell`.

In [ ]:
SLURM_TEMPLATE = textwrap.dedent("""
    #!/bin/bash
    #SBATCH --job-name={job_name}
    #SBATCH -A {account}
    #SBATCH --ntasks={ntasks}
    #SBATCH --cpus-per-task={cpus_per_task}
    #SBATCH -p {partition}
    #SBATCH --mem-per-cpu={mem_per_cpu}
    #SBATCH -t {time}
    #SBATCH --output=slurm_%j.out

    source {venv_activate}
    {python} {script_name}
""").lstrip()

ENERGY_FUNC = textwrap.dedent("""
    # Static FP evaluation (static=True) or full fixed-cell FP relaxation
    # (static=False, atomic positions only -- cell shape/volume held fixed)
    # via ASE FIRE. Raises on any calculator failure; the caller catches
    # that and records status="failed".
    def get_energy(atoms, static=False, fmax=0.01, max_steps=10000):
        atoms = atoms.copy()
        atoms.calc = calc
        if static:
            return {
                "status":      "success",
                "mode":        "static",
                "energy_total": float(atoms.get_potential_energy()),
                "forces":      atoms.get_forces().tolist(),
            }
        from ase.optimize import FIRE
        optimizer = FIRE(atoms, logfile=None)
        converged = optimizer.run(fmax=fmax, steps=max_steps)
        n_steps = optimizer.nsteps
        return {
            "status":            "success" if converged else "non_converged",
            "mode":              "relaxation",
            "energy_total":      float(atoms.get_potential_energy()),
            "forces":            atoms.get_forces().tolist(),
            "relaxed_structure": adaptor.get_structure(atoms).as_dict(),
            "converged":         bool(converged),
            "n_steps":           int(n_steps),
            "fmax_requested":    fmax,
            "max_steps":         max_steps,
            "fixed_cell":        True,
        }
""").strip()

SCRIPT_HEADER_TEMPLATE = textwrap.dedent("""
    #!/usr/bin/env python3
    # Auto-generated by convexhull_ordering_run_generator.ipynb
    # Foundation potential: {mlip_name}
    from __future__ import annotations
    import sys, os, json
    sys.path.insert(0, "{site_pkgs}")

    import numpy as np
    from pymatgen.core import Structure
    from pymatgen.io.ase import AseAtomsAdaptor

    MODEL_PATH = "{model_path}"
    adaptor    = AseAtomsAdaptor()

    {calc_setup}
""").lstrip()


def make_slurm(job_name, script_name, cfg, slurm_cfg):
    return SLURM_TEMPLATE.format(
        job_name      = job_name,
        account       = slurm_cfg["account"],
        ntasks        = slurm_cfg["ntasks"],
        cpus_per_task = slurm_cfg["cpus_per_task"],
        partition     = slurm_cfg["partition"],
        mem_per_cpu   = slurm_cfg["mem_per_cpu"],
        time          = slurm_cfg["time"],
        venv_activate = cfg["venv_activate"],
        python        = cfg["python"],
        script_name   = script_name,
    )


def make_header(cfg):
    return SCRIPT_HEADER_TEMPLATE.format(
        mlip_name  = cfg["mlip_name"],
        site_pkgs  = cfg["site_pkgs"],
        model_path = cfg["model_path"] or "",
        calc_setup = cfg["calc_setup"],
    )


print("Script building blocks ready.")

In [ ]:
from string import Template

CHUNK_SCRIPT_BODY = Template("""
with open("chunk_meta.json") as f:
    chunk_metadata = json.load(f)

with open("mini_dict.json") as f:
    mini_dict = json.load(f)

results = {}

$energy_func

for sid, data in mini_dict.items():
    entry = {k: v for k, v in data.items() if k not in ("initial_structure", "relaxed_structure")}

    try:
        initial_atoms = adaptor.get_atoms(Structure.from_dict(data["initial_structure"]))
        entry["relax"] = get_energy(initial_atoms, static=False, fmax=$fmax, max_steps=$max_steps)
    except Exception as e:
        entry["relax"] = {"status": "failed", "error": str(e)}

    try:
        final_atoms = adaptor.get_atoms(Structure.from_dict(data["relaxed_structure"]))
        entry["static"] = get_energy(final_atoms, static=True)
    except Exception as e:
        entry["static"] = {"status": "failed", "error": str(e)}

    results[sid] = entry
    relax_status = entry["relax"]["status"]
    static_status = entry["static"]["status"]
    print(f"  {sid[:70]:70s}  relax={relax_status:13s}  static={static_status}")

with open("chunk_results.json", "w") as f:
    json.dump({"chunk_metadata": chunk_metadata, "results": results}, f, indent=2)
print(f"Done: {len(results)} candidates -> chunk_results.json")
""")


def calc_setup_sha256(cfg):
    return hashlib.sha256(cfg["calc_setup"].encode("utf-8")).hexdigest()


def write_chunk_job(fp_key, task_name, chunk_idx, mini_dict, cfg, task_dir):
    """Write one chunk's chunk_meta.json + mini_dict.json + job script +
    SLURM script under task_dir/chunk_{chunk_idx:04d}/. chunk_meta.json
    binds this chunk to the exact reference/FP/task/chunk it was generated
    for, so the merge step (Section 8) can verify it later instead of
    trusting it blindly. Returns the SLURM script path (relative to
    task_dir/chunk_.../) -- never submitted here."""
    chunk_id = f"chunk_{chunk_idx:04d}"
    chunk_dir = task_dir / chunk_id
    chunk_dir.mkdir(parents=True, exist_ok=True)

    chunk_meta = {
        "reference_sha256": REFERENCE_SHA256,
        "model_key": fp_key,
        "task": task_name,
        "chunk_id": chunk_id,
        "calculator_setup_sha256": calc_setup_sha256(cfg),
    }
    with open(chunk_dir / "chunk_meta.json", "w") as f:
        json.dump(chunk_meta, f, indent=2)

    with open(chunk_dir / "mini_dict.json", "w") as f:
        json.dump(mini_dict, f)

    script_body = CHUNK_SCRIPT_BODY.substitute(
        energy_func=ENERGY_FUNC, fmax=repr(FMAX), max_steps=repr(MAX_STEPS),
    )
    script = make_header(cfg) + "\n\n" + script_body
    script_name = f"{task_name}_{chunk_id}.py"
    slurm_name = f"{task_name}_{chunk_id}.slurm"

    with open(chunk_dir / script_name, "w") as f:
        f.write(script)
    with open(chunk_dir / slurm_name, "w") as f:
        f.write(make_slurm(f"{fp_key[:8]}_{task_name[:4]}_{chunk_idx:04d}", script_name, cfg, SLURM))
    os.chmod(chunk_dir / script_name, 0o755)

    return chunk_dir / slurm_name


print("write_chunk_job ready.")

---
## Section 5 — Generate hull jobs (interior + endpoint candidates, combined)

Interior candidates and binary endpoints are both drawn from
`reference_data["hull"]` (they already live together, per system, in the
standardized reference -- this is what "endpoints and interior candidates
must be handled together under the hull workflow" means here). A shared
endpoint is computed **once**: its job is generated only for the first
system it appears under; `ENDPOINT_TO_SYSTEMS` records every system it
actually belongs to, so the merge step (Section 8) can route that single
result to every applicable system's `fp_hull` entry instead of recomputing
it once per system.

`sid` keys distinguish the two roles unambiguously for the merge step:
`"{system}||{candidate_id}"` for interior candidates (scoped to their own
system), `"__endpoint__||{candidate_id}"` for endpoints (global identity,
computed once).

In [ ]:
def build_hull_mini_dicts():
    """Returns (mini_dicts, endpoint_to_systems):
    mini_dicts   : list of {sid: {..metadata.., initial_structure, relaxed_structure}}
                   entries (not yet chunked), interior + deduped endpoints.
    endpoint_to_systems : {candidate_id: [system, ...]} for every endpoint,
                   used by Section 8 to route one computed result to every
                   system it borders.
    """
    hull = reference_data["hull"]
    entries = {}
    endpoint_to_systems = defaultdict(list)
    endpoint_seen = set()

    for system in sorted(hull):
        for cid, rec in hull[system].items():
            if rec["role"] == "interior":
                sid = f"{system}||{cid}"
                entries[sid] = {
                    "role": "interior", "system": system, "candidate_id": cid,
                    "phase_id": rec["phase_id"], "composition": rec["composition"],
                    "initial_structure": rec["initial_structure"],
                    "relaxed_structure": rec["relaxed_structure"],
                }
            else:  # endpoint
                endpoint_to_systems[cid].append(system)
                if cid in endpoint_seen:
                    continue
                endpoint_seen.add(cid)
                sid = f"__endpoint__||{cid}"
                entries[sid] = {
                    "role": "endpoint", "candidate_id": cid,
                    "phase_id": rec["phase_id"], "composition": rec["composition"],
                    "endpoint_side": rec["endpoint_side"],
                    "initial_structure": rec["initial_structure"],
                    "relaxed_structure": rec["relaxed_structure"],
                }
    return entries, dict(endpoint_to_systems)


HULL_ENTRIES, ENDPOINT_TO_SYSTEMS = build_hull_mini_dicts()

# Two different, deliberately distinctly-named counts -- never conflate them:
#  - N_HULL_UNIQUE_CANDIDATES: each interior candidate counted once (561) and
#    each endpoint counted once (36) regardless of how many systems it
#    borders. This is what gets *computed* (one job per candidate_id).
#  - N_HULL_SYSTEM_RECORDS: every (system, candidate_id) appearance counted
#    separately, so a shared endpoint counts once per system it borders.
#    This is the size of the fanned-out fp_hull structure
#    build_phase_stability_ordering_results actually consumes, and the
#    correct denominator for any "success" count taken from that structure.
N_HULL_UNIQUE_CANDIDATES = len(HULL_ENTRIES)
N_HULL_SYSTEM_RECORDS = sum(len(candidates) for candidates in reference_data["hull"].values())

print(f"Hull candidates to compute (unique):      {N_HULL_UNIQUE_CANDIDATES} "
      f"({sum(1 for e in HULL_ENTRIES.values() if e['role']=='interior')} interior + "
      f"{sum(1 for e in HULL_ENTRIES.values() if e['role']=='endpoint')} unique endpoints)")
print(f"Hull per-system records (post fan-out):   {N_HULL_SYSTEM_RECORDS} "
      f"(561 interior appearances + {sum(len(s) for s in ENDPOINT_TO_SYSTEMS.values())} endpoint appearances)")
assert N_HULL_UNIQUE_CANDIDATES == 561 + 36
assert N_HULL_SYSTEM_RECORDS == 561 + sum(len(s) for s in ENDPOINT_TO_SYSTEMS.values())
assert N_HULL_SYSTEM_RECORDS == 679

In [ ]:
GENERATED_HULL_SLURM = {}   # fp_key -> [slurm_path, ...]

def generate_hull_jobs(fp_key):
    cfg = POTENTIAL_REGISTRY[fp_key]
    task_dir = OUTPUT_ROOT / fp_key / "hull"
    task_dir.mkdir(parents=True, exist_ok=True)

    sids = sorted(HULL_ENTRIES)
    n_chunks = math.ceil(len(sids) / CHUNK_SIZE)
    slurm_paths = []
    for i in range(n_chunks):
        chunk_sids = sids[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE]
        mini_dict = {sid: HULL_ENTRIES[sid] for sid in chunk_sids}
        slurm_paths.append(write_chunk_job(fp_key, "hull", i, mini_dict, cfg, task_dir))

    GENERATED_HULL_SLURM[fp_key] = slurm_paths
    print(f"  {fp_key:20s}  hull: {len(sids)} candidates -> {n_chunks} chunk(s) under {task_dir}")
    return slurm_paths


print("Generating hull jobs ...")
for fp_key in SELECTED_POTENTIALS:
    generate_hull_jobs(fp_key)

---
## Section 6 — Generate ordering jobs

`reference_data["ordering"]` is keyed by ordering group (one group per
tie-line composition selected for ordering analysis: `305` groups x `20`
orderings/group = `6100` ordering candidates). Every ordering candidate is
distinct to its own group -- unlike hull endpoints, there is no sharing
between groups here, so no deduplication step is needed.

`sid` keys: `"{group_key}||{ordered_name}"`.

In [ ]:
def build_ordering_mini_dicts():
    """Returns a flat {sid: {..metadata.., initial_structure, relaxed_structure}}
    dict, one entry per ordering candidate (group x ordered_name)."""
    ordering = reference_data["ordering"]
    entries = {}
    for group_key, group in ordering.items():
        for ordered_name, rec in group["orderings"].items():
            sid = f"{group_key}||{ordered_name}"
            entries[sid] = {
                "role": "ordering", "group_key": group_key, "ordered_name": ordered_name,
                "system": group["system"], "phase_id": group["phase_id"],
                "composition": group["composition"],
                "initial_structure": rec["initial_structure"],
                "relaxed_structure": rec["relaxed_structure"],
            }
    return entries


ORDERING_ENTRIES = build_ordering_mini_dicts()
print(f"Ordering candidates to compute: {len(ORDERING_ENTRIES)}")
assert len(ORDERING_ENTRIES) == 6100

In [ ]:
GENERATED_ORDERING_SLURM = {}   # fp_key -> [slurm_path, ...]

def generate_ordering_jobs(fp_key):
    cfg = POTENTIAL_REGISTRY[fp_key]
    task_dir = OUTPUT_ROOT / fp_key / "ordering"
    task_dir.mkdir(parents=True, exist_ok=True)

    sids = sorted(ORDERING_ENTRIES)
    n_chunks = math.ceil(len(sids) / CHUNK_SIZE)
    slurm_paths = []
    for i in range(n_chunks):
        chunk_sids = sids[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE]
        mini_dict = {sid: ORDERING_ENTRIES[sid] for sid in chunk_sids}
        slurm_paths.append(write_chunk_job(fp_key, "ordering", i, mini_dict, cfg, task_dir))

    GENERATED_ORDERING_SLURM[fp_key] = slurm_paths
    print(f"  {fp_key:20s}  ordering: {len(sids)} candidates -> {n_chunks} chunk(s) under {task_dir}")
    return slurm_paths


print("Generating ordering jobs ...")
for fp_key in SELECTED_POTENTIALS:
    generate_ordering_jobs(fp_key)

---
## Section 7 — Submit scripts (never executed here)

For every selected FP: one `submit_model.sh` under `OUTPUT_ROOT/{fp}/` that
submits every chunk job (hull + ordering) for that FP with
`sbatch --parsable`, logging one row per submitted job to
`OUTPUT_ROOT/submitted_jobs.tsv`
(`timestamp / registry_key / fp_name / task / chunk_id / SLURM_job_id /
script_path / reference_sha256 / calculator_setup_sha256`). One top-level
`OUTPUT_ROOT/submit_all_models.sh` calls every selected FP's `submit_model.sh`
in turn. Both use `set -euo pipefail`. **Nothing here calls `sbatch` --
these are shell scripts written to disk for the user to run later on
Zaratan; the notebook itself never submits a job.**

In [ ]:
SUBMIT_MODEL_TEMPLATE = Template(textwrap.dedent("""\
    #!/bin/bash
    set -euo pipefail
    # Auto-generated by convexhull_ordering_run_generator.ipynb
    # Submits every hull + ordering chunk job for foundation potential: $fp_key
    # Never executed automatically -- run this manually on Zaratan.
    #
    # Each job is submitted with sbatch run FROM its own chunk directory (via
    # a `cd` inside a $$(...) subshell, which does not affect this script's
    # own working directory) so that SLURM's default working directory for
    # the job matches the chunk directory the job script's relative paths
    # (mini_dict.json, chunk_meta.json, chunk_results.json) assume.

    FP_KEY="$fp_key"
    MLIP_NAME="$mlip_name"
    REFERENCE_SHA256="$reference_sha256"
    CALC_SETUP_SHA256="$calc_setup_sha256"
    HERE="$$(cd "$$(dirname "$${BASH_SOURCE[0]}")" && pwd)"
    LOG_TSV="$${HERE}/../submitted_jobs.tsv"

    log_job() {
      local task="$$1" chunk_id="$$2" job_id="$$3" script_path="$$4"
      printf '%s\\t%s\\t%s\\t%s\\t%s\\t%s\\t%s\\t%s\\t%s\\n' \\
        "$$(date -u +%Y-%m-%dT%H:%M:%SZ)" "$$FP_KEY" "$$MLIP_NAME" "$$task" "$$chunk_id" "$$job_id" "$$script_path" \\
        "$$REFERENCE_SHA256" "$$CALC_SETUP_SHA256" >> "$$LOG_TSV"
    }

    for slurm in "$${HERE}"/hull/chunk_*/hull_chunk_*.slurm; do
      [ -e "$$slurm" ] || continue
      chunk_dir="$$(dirname "$$slurm")"
      chunk_id="$$(basename "$$chunk_dir")"
      job_id="$$(cd "$$chunk_dir" && sbatch --parsable "$$(basename "$$slurm")")"
      log_job "hull" "$$chunk_id" "$$job_id" "$$slurm"
      echo "submitted hull $$chunk_id -> job $$job_id"
    done

    for slurm in "$${HERE}"/ordering/chunk_*/ordering_chunk_*.slurm; do
      [ -e "$$slurm" ] || continue
      chunk_dir="$$(dirname "$$slurm")"
      chunk_id="$$(basename "$$chunk_dir")"
      job_id="$$(cd "$$chunk_dir" && sbatch --parsable "$$(basename "$$slurm")")"
      log_job "ordering" "$$chunk_id" "$$job_id" "$$slurm"
      echo "submitted ordering $$chunk_id -> job $$job_id"
    done
"""))

SUBMIT_ALL_TEMPLATE = Template(textwrap.dedent("""\
    #!/bin/bash
    set -euo pipefail
    # Auto-generated by convexhull_ordering_run_generator.ipynb
    # Submits every selected FP's submit_model.sh in turn.
    # Never executed automatically -- run this manually on Zaratan.

    HERE="$$(cd "$$(dirname "$${BASH_SOURCE[0]}")" && pwd)"

    for fp in $fp_list; do
      echo "=== submitting $$fp ==="
      bash "$${HERE}/$$fp/submit_model.sh"
    done
"""))


def write_submit_scripts(selected_potentials):
    submit_model_paths = {}
    for fp_key in selected_potentials:
        cfg = POTENTIAL_REGISTRY[fp_key]
        fp_dir = OUTPUT_ROOT / fp_key
        fp_dir.mkdir(parents=True, exist_ok=True)
        script = SUBMIT_MODEL_TEMPLATE.substitute(
            fp_key=fp_key, mlip_name=cfg["mlip_name"],
            reference_sha256=REFERENCE_SHA256, calc_setup_sha256=calc_setup_sha256(cfg),
        )
        path = fp_dir / "submit_model.sh"
        with open(path, "w") as f:
            f.write(script)
        os.chmod(path, 0o755)
        submit_model_paths[fp_key] = path

    all_script = SUBMIT_ALL_TEMPLATE.substitute(fp_list=" ".join(selected_potentials))
    all_path = OUTPUT_ROOT / "submit_all_models.sh"
    with open(all_path, "w") as f:
        f.write(all_script)
    os.chmod(all_path, 0o755)

    if not SUBMITTED_JOBS_TSV.exists():
        with open(SUBMITTED_JOBS_TSV, "w") as f:
            f.write("timestamp\tregistry_key\tfp_name\ttask\tchunk_id\tslurm_job_id\tscript_path\treference_sha256\tcalculator_setup_sha256\n")

    return submit_model_paths, all_path


SUBMIT_MODEL_PATHS, SUBMIT_ALL_PATH = write_submit_scripts(SELECTED_POTENTIALS)
print(f"Wrote {len(SUBMIT_MODEL_PATHS)} submit_model.sh script(s) + {SUBMIT_ALL_PATH}")
print(f"Job log (header only, nothing submitted): {SUBMITTED_JOBS_TSV}")

---
## Section 8 — Merge completed chunks and FP fragments

Two steps, matching the pipeline diagram:

1. **Per-FP merge** (`write_fp_fragment(fp_key)`): reads every completed
   `chunk_results.json` under that FP's `hull/` and `ordering/` directories,
   fans each shared endpoint result out to every system it borders (re-
   deriving the endpoint-to-systems mapping from `reference_data["hull"]`
   directly, not from any persisted `ENDPOINT_TO_SYSTEMS`, so this step
   works correctly even if run in a fresh session against only the on-disk
   chunk outputs), and writes one
   `OUTPUT_ROOT/{fp}/standardized_model_fragment.json.gz` containing only
   that FP's own results. **If any candidate's chunk output is missing --
   the run is incomplete -- this raises rather than silently writing a
   partial fragment.** A candidate whose chunk *did* complete but whose FP
   calculation itself failed or did not converge is not an incomplete run;
   its `status` (`failed`/`non_converged`) is preserved as-is.

2. **All-model merge** (`merge_all_fp_fragments(...)`): combines every
   selected FP's fragment via the existing, unmodified
   `merge_phase_stability_ordering_fp_results(...)` into
   `data/phase_stability_ordering_results_standardized.json.gz` -- the same
   function already rejects duplicate model keys, unknown candidate/ordered-
   name identifiers, and reference-checksum mismatches, so none of that is
   reimplemented here. Passing `existing_results_path` extends an
   already-merged file with a genuinely new FP instead of rebuilding it from
   scratch.

In [ ]:
def load_task_chunk_results(fp_key, task):
    """Reads every completed chunk_results.json under
    OUTPUT_ROOT/{fp_key}/{task}/chunk_*/ into one flat {sid: {relax, static}}
    dict. Every chunk's embedded chunk_metadata is verified against the
    currently loaded reference, the requested fp_key/task, the chunk's own
    directory name, and this FP's current calculator-setup checksum before
    its results are trusted -- a stale or mismatched chunk (e.g. computed
    against an older reference, or left over from a different FP or task) is
    rejected outright, never silently merged in under the checksum this call
    happens to be running with. A chunk directory with no chunk_results.json
    yet (job not finished) is simply absent from the result -- completeness
    is checked by the caller against the full expected sid set, not by chunk
    count."""
    cfg = POTENTIAL_REGISTRY[fp_key]
    expected_calc_sha = calc_setup_sha256(cfg)
    task_dir = OUTPUT_ROOT / fp_key / task
    merged = {}
    for chunk_dir in sorted(task_dir.glob("chunk_*")):
        results_path = chunk_dir / "chunk_results.json"
        if not results_path.exists():
            continue
        with open(results_path) as f:
            payload = json.load(f)

        meta = payload.get("chunk_metadata")
        if meta is None:
            raise ValueError(
                f"{fp_key}/{task}/{chunk_dir.name}: chunk_results.json has no chunk_metadata -- "
                "refusing to trust results that are not bound to a known reference/FP/task/chunk"
            )
        problems = []
        if meta.get("reference_sha256") != REFERENCE_SHA256:
            problems.append(f"reference_sha256 {meta.get('reference_sha256')!r} != current {REFERENCE_SHA256!r}")
        if meta.get("model_key") != fp_key:
            problems.append(f"model_key {meta.get('model_key')!r} != expected {fp_key!r}")
        if meta.get("task") != task:
            problems.append(f"task {meta.get('task')!r} != expected {task!r}")
        if meta.get("chunk_id") != chunk_dir.name:
            problems.append(f"chunk_id {meta.get('chunk_id')!r} != directory {chunk_dir.name!r}")
        if meta.get("calculator_setup_sha256") != expected_calc_sha:
            problems.append(
                f"calculator_setup_sha256 {meta.get('calculator_setup_sha256')!r} != "
                f"current {expected_calc_sha!r}"
            )
        if problems:
            raise ValueError(
                f"{fp_key}/{task}/{chunk_dir.name}: stale or mismatched chunk, refusing to merge -- "
                + "; ".join(problems)
            )

        chunk_data = payload["results"]
        overlap = set(chunk_data) & set(merged)
        if overlap:
            raise ValueError(f"{fp_key}/{task}: sid(s) appear in more than one chunk: {sorted(overlap)[:5]}")
        merged.update(chunk_data)
    return merged


def build_fp_fragment(fp_key):
    cfg = POTENTIAL_REGISTRY[fp_key]

    hull_results = load_task_chunk_results(fp_key, "hull")
    ordering_results = load_task_chunk_results(fp_key, "ordering")

    expected_hull_sids = set(HULL_ENTRIES)
    missing_hull = expected_hull_sids - set(hull_results)
    if missing_hull:
        raise ValueError(
            f"{fp_key}: hull run is incomplete -- {len(missing_hull)} candidate(s) have no "
            f"chunk output yet, e.g. {sorted(missing_hull)[:5]}. Refusing to write a fragment "
            "from a partial run."
        )

    expected_ordering_sids = set(ORDERING_ENTRIES)
    missing_ordering = expected_ordering_sids - set(ordering_results)
    if missing_ordering:
        raise ValueError(
            f"{fp_key}: ordering run is incomplete -- {len(missing_ordering)} candidate(s) have "
            f"no chunk output yet, e.g. {sorted(missing_ordering)[:5]}. Refusing to write a "
            "fragment from a partial run."
        )

    # ── hull: fan each shared endpoint result out to every system it borders ──
    hull_frag = {"relax": defaultdict(dict), "static": defaultdict(dict)}
    for system, candidates in reference_data["hull"].items():
        for cid, rec in candidates.items():
            sid = f"{system}||{cid}" if rec["role"] == "interior" else f"__endpoint__||{cid}"
            entry = hull_results[sid]
            hull_frag["relax"][system][cid] = dict(entry["relax"])
            hull_frag["static"][system][cid] = dict(entry["static"])
    hull_frag = {"relax": dict(hull_frag["relax"]), "static": dict(hull_frag["static"])}

    # ── ordering: one-to-one, no fan-out needed ────────────────────────────────
    ordering_frag = {"relax": defaultdict(dict), "static": defaultdict(dict)}
    for sid, entry in ordering_results.items():
        group_key, ordered_name = sid.split("||", 1)
        ordering_frag["relax"][group_key][ordered_name] = dict(entry["relax"])
        ordering_frag["static"][group_key][ordered_name] = dict(entry["static"])
    ordering_frag = {"relax": dict(ordering_frag["relax"]), "static": dict(ordering_frag["static"])}

    def n_success_unique(results_dict, mode):
        """Unique-candidate view: each interior candidate counted once (561)
        and each endpoint counted once (36) regardless of how many systems
        it borders -- out of N_HULL_UNIQUE_CANDIDATES (597), matching
        HULL_ENTRIES / the pre-fan-out hull_results."""
        return sum(1 for entry in results_dict.values() if entry[mode].get("status") == "success")

    def n_success_records(frag, mode):
        """Per-system-record view: every (system, candidate_id) appearance
        counted separately, so a shared endpoint is counted once per system
        it borders -- out of N_HULL_SYSTEM_RECORDS (679), matching the
        fp_hull structure fed to build_phase_stability_ordering_results."""
        return sum(1 for sysdict in frag[mode].values() for rec in sysdict.values() if rec.get("status") == "success")

    def n_ordering_success(frag, mode):
        return sum(1 for groupdict in frag[mode].values() for rec in groupdict.values() if rec.get("status") == "success")

    fragment = {
        "schema_version": reference_payload["schema_version"],
        "dataset_name": reference_payload["dataset_name"],
        "component": "phase_stability_ordering",
        "reference": {"sha256": REFERENCE_SHA256, "path": str(REFERENCE_PATH)},
        "model_key": fp_key,
        "metadata": {
            "registry_key": fp_key,
            "mlip_name": cfg["mlip_name"],
            "model_path": cfg["model_path"],
            "site_pkgs": cfg["site_pkgs"],
            "venv_activate": cfg["venv_activate"],
            "python": cfg["python"],
            "calculator_setup_sha256": calc_setup_sha256(cfg),
            "reference_sha256": REFERENCE_SHA256,
            "generator_notebook": "convexhull_ordering_run_generator.ipynb",
            "protocol": {
                "relax": {
                    "label": "Full FP relaxation",
                    "optimizer": "ASE FIRE", "fmax": FMAX, "max_steps": MAX_STEPS,
                    "fixed_cell": FIXED_CELL, "starting_geometry": "initial_structure",
                },
                "static": {
                    "label": "Static FP evaluations on the DFT-relaxed structures",
                    "starting_geometry": "relaxed_structure",
                },
            },
            "counts": {
                "hull_unique_candidates_expected": len(expected_hull_sids),
                "hull_unique_candidates_relax_success": n_success_unique(hull_results, "relax"),
                "hull_unique_candidates_static_success": n_success_unique(hull_results, "static"),
                "hull_system_records_expected": N_HULL_SYSTEM_RECORDS,
                "hull_system_records_relax_success": n_success_records(hull_frag, "relax"),
                "hull_system_records_static_success": n_success_records(hull_frag, "static"),
                "ordering_expected": len(expected_ordering_sids),
                "ordering_relax_success": n_ordering_success(ordering_frag, "relax"),
                "ordering_static_success": n_ordering_success(ordering_frag, "static"),
            },
            "merged_utc": datetime.now(timezone.utc).isoformat(),
        },
        "hull": hull_frag,
        "ordering": ordering_frag,
    }
    return fragment


def write_fp_fragment(fp_key):
    fragment = build_fp_fragment(fp_key)
    out_path = OUTPUT_ROOT / fp_key / "standardized_model_fragment.json.gz"
    write_standardized_json_gz(out_path, fragment)
    c = fragment["metadata"]["counts"]
    hull_unique_expected = c["hull_unique_candidates_expected"]
    hull_unique_relax = c["hull_unique_candidates_relax_success"]
    hull_unique_static = c["hull_unique_candidates_static_success"]
    hull_records_expected = c["hull_system_records_expected"]
    hull_records_relax = c["hull_system_records_relax_success"]
    hull_records_static = c["hull_system_records_static_success"]
    ordering_expected = c["ordering_expected"]
    ordering_relax_success = c["ordering_relax_success"]
    ordering_static_success = c["ordering_static_success"]
    print(f"  {fp_key:20s}  wrote {out_path}")
    print(f"    hull (unique candidates):  relax {hull_unique_relax}/{hull_unique_expected} success   "
          f"static {hull_unique_static}/{hull_unique_expected} success")
    print(f"    hull (per-system records): relax {hull_records_relax}/{hull_records_expected} success   "
          f"static {hull_records_static}/{hull_records_expected} success")
    print(f"    ordering:                  relax {ordering_relax_success}/{ordering_expected} success   "
          f"static {ordering_static_success}/{ordering_expected} success")
    return out_path


print("build_fp_fragment / write_fp_fragment ready.")

In [ ]:
MERGED_RESULTS_PATH = Path("data/phase_stability_ordering_results_standardized.json.gz")


def merge_all_fp_fragments(selected_potentials, existing_results_path=None, out_path=None):
    """Combine every selected FP's standardized_model_fragment.json.gz into
    one all-model results file via the existing, unmodified
    merge_phase_stability_ordering_fp_results. Pass existing_results_path to
    add genuinely new FP(s) to an already-merged file instead of rebuilding
    it from scratch -- duplicate model keys, unknown identifiers, and
    reference-checksum mismatches are all rejected by that function itself,
    not reimplemented here."""
    fragments = {}
    for fp_key in selected_potentials:
        frag_path = OUTPUT_ROOT / fp_key / "standardized_model_fragment.json.gz"
        if not frag_path.exists():
            raise FileNotFoundError(
                f"{fp_key}: no fragment at {frag_path} -- run write_fp_fragment({fp_key!r}) first"
            )
        fragments[fp_key] = str(frag_path)

    existing_models = None
    if existing_results_path is not None and Path(existing_results_path).exists():
        existing_models = load_standardized_results(Path(existing_results_path))

    merged = merge_phase_stability_ordering_fp_results(
        fragments, reference_data,
        schema_version=reference_payload["schema_version"],
        dataset_name=reference_payload["dataset_name"],
        reference_sha256=REFERENCE_SHA256,
        existing_models=existing_models,
        out_path=out_path,
    )
    return merged


print("merge_all_fp_fragments ready.")

---
## Section 9 — Finalize and merge completed results (guarded, disabled by default)

Writes each selected FP's fragment and merges the selected fragments into
`data/phase_stability_ordering_results_standardized.json.gz` for real. Two
independent guards keep a routine "Run All" from ever touching the canonical
file:

- `RUN_REAL_MERGE` defaults to `False` -- nothing below runs at all.
- `MERGE_MODE` must be set explicitly to distinguish a **fresh build**
  (`"fresh_build"`, refuses if the merged file already exists) from
  **adding a genuinely new FP** to an already-merged file (`"add_new_fp"`,
  refuses if it does not exist yet). `merge_phase_stability_ordering_fp_results`
  itself still rejects any duplicate model key on top of that.

Only set `RUN_REAL_MERGE = True` after confirming every selected FP's chunks
have actually finished (e.g. via `submitted_jobs.tsv` and checking each
chunk directory for `chunk_results.json`) -- `write_fp_fragment` raises on
any FP whose run is still incomplete (Section 8), and `load_task_chunk_results`
rejects any chunk whose embedded metadata does not match the currently
loaded reference (Section 8).

In [ ]:
RUN_REAL_MERGE = False        # set True only after confirming every selected FP's chunks are done
MERGE_MODE = "fresh_build"    # "fresh_build" or "add_new_fp" -- only read when RUN_REAL_MERGE is True

if RUN_REAL_MERGE:
    if MERGE_MODE not in ("fresh_build", "add_new_fp"):
        raise ValueError(f"MERGE_MODE must be \'fresh_build\' or \'add_new_fp\', got {MERGE_MODE!r}")

    written_fragments = {fp_key: write_fp_fragment(fp_key) for fp_key in SELECTED_POTENTIALS}

    if MERGE_MODE == "fresh_build":
        if MERGED_RESULTS_PATH.exists():
            raise FileExistsError(
                f"{MERGED_RESULTS_PATH} already exists -- \'fresh_build\' would ignore its "
                "existing models. Use MERGE_MODE=\'add_new_fp\' to extend it instead, or "
                "move/remove the existing file first if a full rebuild is really intended."
            )
        existing_path = None
    else:
        if not MERGED_RESULTS_PATH.exists():
            raise FileNotFoundError(
                f"{MERGED_RESULTS_PATH} does not exist yet -- use MERGE_MODE=\'fresh_build\' first."
            )
        existing_path = MERGED_RESULTS_PATH

    merged = merge_all_fp_fragments(SELECTED_POTENTIALS, existing_results_path=existing_path,
                                     out_path=MERGED_RESULTS_PATH)
    merged_models = list(merged["models"])
    print(f"Wrote {len(written_fragments)} fragment(s) and merged into {MERGED_RESULTS_PATH} "
          f"(mode={MERGE_MODE}). Models now in the merged file: {merged_models}")
else:
    print(f"RUN_REAL_MERGE is False -- nothing written, {MERGED_RESULTS_PATH} untouched. "
          "Set RUN_REAL_MERGE = True and choose MERGE_MODE only after confirming every "
          "selected FP\'s chunks have finished on Zaratan.")